# BYV-AI — BYV-FP v0.2.1 QLoRA Training

**Base model:** `Qwen/Qwen3-4B-Thinking-2507`  
**Dataset:** BYV-FP v0.2.1 (280 train / 35 eval; held-out test is not downloaded here)  
**Method:** 4-bit NF4 QLoRA + LoRA adapter  
**Target hardware:** Colab T4 as a feasibility run (FP16; batch 1; gradient checkpointing)

Notebook ini melatih model menghasilkan jawaban biasa atau BYV-FP sebagai teks. Ia **tidak** mengeksekusi tool. Tool schema dipasok di system message dan semua target tool-call divalidasi terhadap schema tersebut.

> **Penting:** dataset ini hasil migrasi mekanis dari seed sintetis v0.1.1. Struktur dan schema tool sudah lolos validasi otomatis, tetapi semantic correctness dan kualitas semua kode masih perlu review manusia. Jangan anggap ini benchmark SOTA.

Jalankan cell dari atas ke bawah di Google Colab dengan GPU. Kalau T4 kehabisan VRAM, turunkan `MAX_LENGTH` ke `512` sebelum menjalankan training.

In [ ]:
%pip install -q -U "transformers>=4.51.0" "accelerate>=1.0.0" "datasets>=3.0.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" "safetensors>=0.4.5"

In [ ]:
import os, sys, json, re, time, math, hashlib, random, inspect, platform, subprocess
from pathlib import Path
from collections import Counter

import torch
import transformers
from transformers import AutoTokenizer
from datasets import Dataset

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
from packaging.version import Version
if Version(torch.__version__.split('+')[0]) < Version("2.6.0"):
    raise RuntimeError("This Qwen3-4B-Thinking-2507 notebook requires PyTorch >= 2.6.0. Select a Colab runtime with a newer CUDA-enabled PyTorch build; do not upgrade torch blindly inside an already configured CUDA runtime.")


In [ ]:
# ---------- Experiment config ----------
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
MODEL_REVISION = "768f209d9ea81521153ed38c47d515654e938aea"  # pinned HF revision
DATASET_VERSION = "byv-fp-v0.2.1"
EXPECTED_DATA_SHA256 = {
    "train.jsonl": "263deb0927fde69d34e8186329bf8624727eac47a72a3a6845bd8e5125b0bcb1",
    "eval.jsonl": "c23166f3241fcf908fd9c675fe3e6cde8cbefe4d96f65723c8fa238d50b42373",
}

SEED = 42
MAX_LENGTH = 1024
NUM_EPOCHS = 2
LEARNING_RATE = 1.0e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_BASELINE_NEW_TOKENS = 1024
MIN_TRAIN_RECORDS = 200
MIN_TRAIN_TURNS = 200

RUN_NAME = "byv-ai-qwen3-4b-thinking-2507-byvfp-qlora-t4-fp16-v0.2.1"
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Model:", MODEL_ID)
print("Dataset:", DATASET_VERSION)
print("Run:", RUN_NAME)

## 1. Mount Google Drive dan siapkan dataset (upload satu kali)

Cell ini memakai dataset ZIP yang disimpan di Google Drive. Pada pemakaian pertama, upload ZIP BYV-FP v0.2.1 sekali saja; file akan disimpan ke Drive dan dipakai ulang saat training/evaluasi berikutnya. Notebook training hanya membaca `train.jsonl` dan `eval.jsonl`, tidak membuka `test.jsonl`.

In [ ]:
from google.colab import drive, files
import io, zipfile

ZIP_NAME = "BYV-AI_dataset_BYV-FP_v0.2.1.zip"
ZIP_ROOT = "BYV-AI_dataset_byvfp_v0.2.1/"

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/BYV-AI')
DATA_DIR = DRIVE_ROOT / 'data' / DATASET_VERSION
OUTPUT_DIR = DRIVE_ROOT / 'outputs' / RUN_NAME
EXPERIMENT_DIR = DRIVE_ROOT / 'experiments' / RUN_NAME
for folder in (DATA_DIR, OUTPUT_DIR, EXPERIMENT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DATA_DIR / ZIP_NAME
if not ZIP_PATH.exists():
    print(f"Upload {ZIP_NAME} sekali saja. Salinannya akan disimpan di Google Drive.")
    uploaded = files.upload()
    if ZIP_NAME not in uploaded:
        raise FileNotFoundError(f"Please upload the exact file: {ZIP_NAME}")
    ZIP_PATH.write_bytes(uploaded[ZIP_NAME])
    del uploaded
else:
    print("Dataset ZIP ditemukan di Google Drive; memakai salinan yang sudah ada.")

with zipfile.ZipFile(ZIP_PATH) as archive:
    members = set(archive.namelist())
    for filename, expected_hash in EXPECTED_DATA_SHA256.items():
        member = ZIP_ROOT + filename
        if member not in members:
            raise FileNotFoundError(f"Dataset ZIP is missing {member}")
        payload = archive.read(member)  # deliberately read train/eval only; test.jsonl is never opened here
        actual_hash = hashlib.sha256(payload).hexdigest()
        if actual_hash != expected_hash:
            raise RuntimeError(f"SHA-256 mismatch for {filename}: {actual_hash} != {expected_hash}")
        (DATA_DIR / filename).write_bytes(payload)
        print(f"Verified {filename}: {actual_hash}")

TRAIN_PATH = DATA_DIR / 'train.jsonl'
EVAL_PATH = DATA_DIR / 'eval.jsonl'
print("Train:", TRAIN_PATH)
print("Eval:", EVAL_PATH)
print("Adapter output:", OUTPUT_DIR)

## 2. Periksa dataset dan BYV-FP schema

Validator ini memeriksa envelope, tool allowlist, required fields, tipe JSON, dan properti tambahan. Ini validasi struktural/schema; bukan pembuktian kebenaran semantik jawaban.

In [ ]:
HEADER = "BYV-FP/1.0"
ID_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._:-]{0,63}$")
TOOL_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_.:-]{0,127}$")


def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"{path}:{line_no}: invalid JSON: {exc}") from exc
    return rows


def get_active_tools(messages):
    system = next((m["content"] for m in messages if m["role"] == "system"), "")
    marker = "ACTIVE_TOOLS_JSON:\n"
    if marker not in system:
        return {}
    block = system.split(marker, 1)[1].split("\nUse the schemas above", 1)[0]
    return {t["name"]: t["input_schema"] for t in json.loads(block)}


def schema_errors(value, schema, path="$" ):
    errors = []
    typ = schema.get("type")
    checks = {
        "object": lambda x: isinstance(x, dict),
        "array": lambda x: isinstance(x, list),
        "string": lambda x: isinstance(x, str),
        "integer": lambda x: isinstance(x, int) and not isinstance(x, bool),
        "number": lambda x: isinstance(x, (int, float)) and not isinstance(x, bool),
        "boolean": lambda x: isinstance(x, bool),
        "null": lambda x: x is None,
    }
    if typ and (typ not in checks or not checks[typ](value)):
        return [f"{path}: expected {typ}, got {type(value).__name__}"]
    if isinstance(value, dict):
        props = schema.get("properties", {})
        for key in schema.get("required", []):
            if key not in value:
                errors.append(f"{path}.{key}: missing required property")
        for key, item in value.items():
            if key not in props:
                if schema.get("additionalProperties", True) is False:
                    errors.append(f"{path}.{key}: undeclared property")
            else:
                errors.extend(schema_errors(item, props[key], f"{path}.{key}"))
    if isinstance(value, list) and "items" in schema:
        for i, item in enumerate(value):
            errors.extend(schema_errors(item, schema["items"], f"{path}[{i}]"))
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        if "minimum" in schema and value < schema["minimum"]:
            errors.append(f"{path}: below minimum")
        if "maximum" in schema and value > schema["maximum"]:
            errors.append(f"{path}: above maximum")
    if isinstance(value, str):
        if "minLength" in schema and len(value) < schema["minLength"]:
            errors.append(f"{path}: shorter than minLength")
        if "maxLength" in schema and len(value) > schema["maxLength"]:
            errors.append(f"{path}: longer than maxLength")
    return errors


def validate_target(text, messages):
    if not text.startswith(HEADER + "\n"):
        if "｜DSML｜" in text or "<｜DSML｜" in text or "</｜DSML｜" in text:
            return ["legacy DSML marker remains in assistant target"]
        return []
    tools = get_active_tools(messages)
    try:
        envelope = json.loads(text.split("\n", 1)[1])
    except Exception as exc:
        return [f"invalid BYV-FP JSON: {exc}"]
    if set(envelope) != {"type", "calls"} or envelope.get("type") != "tool_calls":
        return ["invalid BYV-FP top-level envelope"]
    calls = envelope.get("calls")
    if not isinstance(calls, list) or not (1 <= len(calls) <= 32):
        return ["calls must be a non-empty array with at most 32 items"]
    errors = []
    seen = set()
    for i, call in enumerate(calls):
        if not isinstance(call, dict) or set(call) != {"id", "tool", "arguments"}:
            errors.append(f"calls[{i}]: keys must be id, tool, arguments")
            continue
        call_id, tool, arguments = call["id"], call["tool"], call["arguments"]
        if not isinstance(call_id, str) or not ID_RE.fullmatch(call_id):
            errors.append(f"calls[{i}]: invalid id")
        if call_id in seen:
            errors.append(f"calls[{i}]: duplicate id")
        seen.add(call_id)
        if not isinstance(tool, str) or not TOOL_RE.fullmatch(tool):
            errors.append(f"calls[{i}]: invalid tool name")
            continue
        if tool not in tools:
            errors.append(f"calls[{i}]: unknown tool {tool}")
            continue
        if not isinstance(arguments, dict):
            errors.append(f"calls[{i}]: arguments must be an object")
            continue
        errors.extend(schema_errors(arguments, tools[tool], f"calls[{i}].arguments"))
    return errors


def validate_records(rows, label):
    ids = [r.get("id") for r in rows]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate record IDs in {label}")
    counts = Counter(r.get("category", "uncategorized") for r in rows)
    tool_targets = 0
    for row in rows:
        if not isinstance(row.get("messages"), list) or not row["messages"]:
            raise ValueError(f"{label}/{row.get('id')}: missing messages")
        if row["messages"][0]["role"] != "system":
            raise ValueError(f"{label}/{row.get('id')}: first message must be system")
        for i, msg in enumerate(row["messages"]):
            if msg["role"] not in {"system", "user", "assistant"}:
                raise ValueError(f"{label}/{row['id']}: unsupported role {msg['role']}")
            if msg["role"] == "assistant":
                errs = validate_target(msg["content"], row["messages"][:i])
                if errs:
                    raise ValueError(f"{label}/{row['id']} assistant[{i}]: {errs}")
                if msg["content"].startswith(HEADER + "\n"):
                    tool_targets += 1
            if "DSML" in msg["content"] or "｜DSML｜" in msg["content"]:
                raise ValueError(f"{label}/{row['id']}: legacy DSML term found")
    return counts, tool_targets

TRAIN_ROWS = load_jsonl(TRAIN_PATH)
EVAL_ROWS = load_jsonl(EVAL_PATH)
train_counts, train_tool_targets = validate_records(TRAIN_ROWS, "train")
eval_counts, eval_tool_targets = validate_records(EVAL_ROWS, "eval")
if len(TRAIN_ROWS) < MIN_TRAIN_RECORDS:
    raise RuntimeError(f"Expected at least {MIN_TRAIN_RECORDS} train records; got {len(TRAIN_ROWS)}")
if {r['id'] for r in TRAIN_ROWS} & {r['id'] for r in EVAL_ROWS}:
    raise RuntimeError("Train/eval record ID overlap detected")

print("Train records:", len(TRAIN_ROWS), "| tool-call assistant messages:", train_tool_targets)
print("Eval records:", len(EVAL_ROWS), "| tool-call assistant messages:", eval_tool_targets)
print("Train categories:", dict(train_counts))
print("Eval categories:", dict(eval_counts))
print("All train/eval records passed BYV-FP structure + active-schema validation.")

## 3. Periksa GPU dan dtype

Tesla T4 (compute capability 7.5) menggunakan FP16, bukan BF16. Notebook menentukan dtype dari compute capability GPU yang aktif.

In [ ]:
GPU_AVAILABLE = torch.cuda.is_available()
GPU_NAME = None
GPU_VRAM_GB = None
GPU_COMPUTE_CAPABILITY = None
if GPU_AVAILABLE:
    props = torch.cuda.get_device_properties(0)
    GPU_NAME = props.name
    GPU_VRAM_GB = round(props.total_memory / (1024**3), 2)
    GPU_COMPUTE_CAPABILITY = torch.cuda.get_device_capability(0)
    native_bf16 = GPU_COMPUTE_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
    COMPUTE_DTYPE = torch.bfloat16 if native_bf16 else torch.float16
    print("GPU:", GPU_NAME)
    print("VRAM:", GPU_VRAM_GB, "GB")
    print("Compute capability:", GPU_COMPUTE_CAPABILITY)
    print("Compute dtype:", COMPUTE_DTYPE)
else:
    COMPUTE_DTYPE = torch.float16
    print("No CUDA GPU. Data validation is available, but QLoRA training will be skipped.")

## 4. Tokenizer dan chat-template probe

Model Thinking-2507 selalu memakai thinking mode. Karena dataset ini hanya menyediakan jawaban final (bukan hidden reasoning traces), setiap target assistant diberi `</think>` di awal untuk menutup blok thinking kosong dan menempatkan label di final-answer channel. Tidak ada private chain-of-thought yang dibuat atau disimpan.

In [ ]:
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=True)
if TOKENIZER.pad_token_id is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.padding_side = "right"

probe_messages = [
    {"role": "system", "content": "You are BYV-AI. Answer plainly."},
    {"role": "user", "content": "Jelaskan fungsi README dalam satu kalimat."},
]
probe_text = TOKENIZER.apply_chat_template(probe_messages, tokenize=False, add_generation_prompt=True)
print("Chat-template suffix:", repr(probe_text[-100:]))
if "<think>" not in probe_text:
    raise RuntimeError("Qwen3-4B-Thinking-2507 template did not produce the expected <think> generation prefix")


def final_only_content(content):
    # Dataset targets are final-only. Close the template's empty thinking block.
    if "</think>" in content or "<think>" in content:
        return content
    return "</think>\n" + content


def normalize_context_messages(messages):
    out = []
    for msg in messages:
        item = {"role": msg["role"], "content": msg["content"]}
        if item["role"] == "assistant":
            item["content"] = final_only_content(item["content"])
        out.append(item)
    return out

print("Thinking template probe passed. No reasoning traces are part of this dataset.")

## 5. Ubah setiap assistant turn menjadi contoh causal-LM dengan loss hanya pada target

Setiap jawaban assistant menjadi satu training sample. Token prompt diberi label `-100`, jadi loss dihitung pada jawaban assistant saja. Riwayat multi-turn tetap menjadi konteks. Jika satu jawaban target terlalu panjang untuk `MAX_LENGTH`, sample tersebut dilewati dan jumlahnya dilaporkan; prompt dipotong dari kiri agar target tidak terpotong.

In [ ]:
def token_ids(messages, add_generation_prompt):
    result = TOKENIZER.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=add_generation_prompt,
    )
    if isinstance(result, dict):
        result = result["input_ids"]
    if isinstance(result, torch.Tensor):
        result = result.tolist()
    return list(result)


def expand_assistant_turns(rows, split_name):
    samples = []
    skipped_long_targets = []
    for row in rows:
        messages = row["messages"]
        for i, msg in enumerate(messages):
            if msg["role"] != "assistant":
                continue
            prefix = messages[:i]
            if not any(m["role"] == "user" for m in prefix):
                continue
            prefix_messages = normalize_context_messages(prefix)
            target_content = final_only_content(msg["content"])
            full_messages = prefix_messages + [{"role": "assistant", "content": target_content}]

            prompt_ids = token_ids(prefix_messages, add_generation_prompt=True)
            full_ids = token_ids(full_messages, add_generation_prompt=False)

            if full_ids[:len(prompt_ids)] != prompt_ids:
                raise RuntimeError(
                    f"Chat-template prefix mismatch for {split_name}/{row['id']} assistant[{i}]. "
                    "Check Qwen tokenizer template or message normalization."
                )
            target_ids = full_ids[len(prompt_ids):]
            if len(target_ids) >= MAX_LENGTH or len(full_ids) > MAX_LENGTH:
                skipped_long_targets.append({
                    "id": row["id"],
                    "target_tokens": len(target_ids),
                    "total_tokens": len(full_ids),
                })
                continue
            # Preserve the entire trusted system/tool schema and conversation; never truncate its prefix.
            input_ids = full_ids
            labels = [-100] * len(prompt_ids) + target_ids
            samples.append({
                "input_ids": input_ids,
                "labels": labels,
                "attention_mask": [1] * len(input_ids),
                "record_id": row["id"],
                "category": row.get("category", "uncategorized"),
            })
    return samples, skipped_long_targets

TRAIN_SAMPLES, TRAIN_SKIPPED = expand_assistant_turns(TRAIN_ROWS, "train")
EVAL_SAMPLES, EVAL_SKIPPED = expand_assistant_turns(EVAL_ROWS, "eval")
if len(TRAIN_SAMPLES) < MIN_TRAIN_TURNS or len(EVAL_SAMPLES) < 10:
    raise RuntimeError(
        f"Too few assistant turns after tokenization: train={len(TRAIN_SAMPLES)}, eval={len(EVAL_SAMPLES)}"
    )

print("Train assistant-turn samples:", len(TRAIN_SAMPLES))
print("Eval assistant-turn samples:", len(EVAL_SAMPLES))
print("Skipped long train targets:", len(TRAIN_SKIPPED))
print("Skipped long eval targets:", len(EVAL_SKIPPED))
if TRAIN_SKIPPED:
    print("First skipped train target(s):", TRAIN_SKIPPED[:10])
if EVAL_SKIPPED:
    print("First skipped eval target(s):", EVAL_SKIPPED[:10])

In [ ]:
TRAIN_DATASET = Dataset.from_list(TRAIN_SAMPLES)
EVAL_DATASET = Dataset.from_list(EVAL_SAMPLES)

class BYVDataCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)
        input_ids, attention_masks, labels = [], [], []
        for item in features:
            pad = max_len - len(item["input_ids"])
            input_ids.append(item["input_ids"] + [self.pad_token_id] * pad)
            attention_masks.append(item["attention_mask"] + [0] * pad)
            labels.append(item["labels"] + [-100] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

COLLATOR = BYVDataCollator(TOKENIZER.pad_token_id)
print("Prepared train/eval datasets with assistant-only labels.")

## 6. Load 4-bit base model dan pasang LoRA

T4 akan memakai NF4 + double quantization + FP16 compute. Jika model gagal dimuat karena VRAM, coba runtime dengan GPU lebih besar atau turunkan `MAX_LENGTH` menjadi `512` lalu restart dan jalankan ulang dari awal.

In [ ]:
MODEL_READY = False
model = None
trainer = None
BASE_MODEL_COMMIT = None

if GPU_AVAILABLE:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    major_version = int(transformers.__version__.split(".")[0])
    dtype_key = "dtype" if major_version >= 5 else "torch_dtype"
    load_kwargs = {
        "revision": MODEL_REVISION,
        "quantization_config": bnb_config,
        "device_map": "auto",
        "low_cpu_mem_usage": True,
    }
    load_kwargs[dtype_key] = COMPUTE_DTYPE

    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
    except TypeError as exc:
        # Compatibility fallback for a transformers build that expects torch_dtype.
        if dtype_key == "dtype":
            load_kwargs.pop("dtype", None)
            load_kwargs["torch_dtype"] = COMPUTE_DTYPE
            model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
        else:
            raise exc

    BASE_MODEL_COMMIT = getattr(model.config, "_commit_hash", None)
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)

    # Keep trainable adapter weights in FP32 for stable FP16 training on T4.
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.dtype != torch.float32:
            parameter.data = parameter.data.float()

    model.print_trainable_parameters()
    trainable_dtypes = Counter(str(p.dtype) for p in model.parameters() if p.requires_grad)
    print("Trainable parameter dtypes:", dict(trainable_dtypes))
    if any(p.dtype != torch.float32 for p in model.parameters() if p.requires_grad):
        raise RuntimeError("Trainable LoRA parameters are not all FP32; stopping before training.")
    MODEL_READY = True
else:
    print("MODEL_READY=False because CUDA is unavailable.")

## 7. Jalankan SFT QLoRA

Training memakai `Trainer` langsung supaya loss masking assistant-only eksplisit dan tidak bergantung pada perubahan API `SFTTrainer`. Dependency `trl` tidak diperlukan.

In [ ]:
TRAINING_READY = bool(MODEL_READY and len(TRAIN_DATASET) >= MIN_TRAIN_TURNS and len(EVAL_DATASET) >= 10)
if TRAINING_READY:
    from transformers import TrainingArguments, Trainer

    args_signature = inspect.signature(TrainingArguments.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in args_signature else "evaluation_strategy"
    training_kwargs = {
        "output_dir": str(OUTPUT_DIR),
        "run_name": RUN_NAME,
        "num_train_epochs": NUM_EPOCHS,
        "per_device_train_batch_size": PER_DEVICE_BATCH_SIZE,
        "per_device_eval_batch_size": 1,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": 0.03,
        "lr_scheduler_type": "cosine",
        "weight_decay": 0.01,
        "optim": "paged_adamw_8bit",
        "logging_steps": 5,
        "save_strategy": "epoch",
        "save_total_limit": 2,
        "fp16": COMPUTE_DTYPE == torch.float16,
        "bf16": COMPUTE_DTYPE == torch.bfloat16,
        "gradient_checkpointing": True,
        "max_grad_norm": 0.3,
        "report_to": "none",
        "remove_unused_columns": False,
        "dataloader_pin_memory": False,
        "seed": SEED,
        "data_seed": SEED,
    }
    training_kwargs[eval_key] = "epoch"
    if "gradient_checkpointing_kwargs" in args_signature:
        training_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

    training_args = TrainingArguments(**training_kwargs)
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=TRAIN_DATASET,
        eval_dataset=EVAL_DATASET,
        data_collator=COLLATOR,
    )
    print("Trainer ready. GPU:", GPU_NAME, "| VRAM:", GPU_VRAM_GB, "GB")
else:
    print("Training not ready. Confirm CUDA, model load, and dataset counts above.")

In [ ]:
if TRAINING_READY:
    try:
        train_result = trainer.train()
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower() or "cuda error" in str(exc).lower():
            torch.cuda.empty_cache()
            raise RuntimeError(
                "CUDA/VRAM failure. Restart runtime, set MAX_LENGTH=512, keep batch=1, "
                "and run cells from the beginning. A larger GPU may be required."
            ) from exc
        raise

    trainer.save_model(str(OUTPUT_DIR))
    TOKENIZER.save_pretrained(str(OUTPUT_DIR))

    metrics = dict(train_result.metrics)
    metrics.update({
        "train_records": len(TRAIN_ROWS),
        "eval_records": len(EVAL_ROWS),
        "train_assistant_turns": len(TRAIN_DATASET),
        "eval_assistant_turns": len(EVAL_DATASET),
        "skipped_long_train_targets": len(TRAIN_SKIPPED),
        "skipped_long_eval_targets": len(EVAL_SKIPPED),
        "base_model": MODEL_ID,
        "base_model_revision": BASE_MODEL_COMMIT,
        "dataset_version": DATASET_VERSION,
        "train_sha256": EXPECTED_DATA_SHA256["train.jsonl"],
        "eval_sha256": EXPECTED_DATA_SHA256["eval.jsonl"],
        "gpu_name": GPU_NAME,
        "gpu_vram_gb": GPU_VRAM_GB,
        "compute_dtype": str(COMPUTE_DTYPE),
        "max_length": MAX_LENGTH,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
    })
    (EXPERIMENT_DIR / "train_metrics.json").write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8"
    )
    (EXPERIMENT_DIR / "trainer_log_history.json").write_text(
        json.dumps(trainer.state.log_history, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8"
    )
    print("Adapter saved to:", OUTPUT_DIR)
    print(json.dumps(metrics, ensure_ascii=False, indent=2, default=str))
else:
    print("Training skipped because the readiness gate was not satisfied.")

## 8. Smoke test pada contoh eval (bukan held-out test)

Cell ini menggunakan sebagian kecil dari `eval.jsonl` untuk melihat bentuk output setelah training. Ia tidak membaca `test.jsonl` dan tidak menjalankan tool.

In [ ]:
def extract_final_channel(generated_text: str) -> str:
    # Do not print/save private thinking content. Return only text after final delimiter.
    if "</think>" in generated_text:
        return generated_text.rsplit("</think>", 1)[1].strip()
    if "<think>" in generated_text:
        return "[NO_FINAL_CHANNEL_DELIMITER]"
    return generated_text.strip()


def generate_final(model_obj, messages, max_new_tokens=MAX_BASELINE_NEW_TOKENS):
    prompt_messages = normalize_context_messages(messages)
    prompt = TOKENIZER.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    device = model_obj.get_input_embeddings().weight.device
    inputs = TOKENIZER(prompt, return_tensors="pt").to(device)
    model_obj.config.use_cache = True
    with torch.inference_mode():
        output = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=TOKENIZER.pad_token_id,
            eos_token_id=TOKENIZER.eos_token_id,
            use_cache=True,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    continuation = TOKENIZER.decode(new_tokens, skip_special_tokens=True)
    model_obj.config.use_cache = False
    return extract_final_channel(continuation)


if TRAINING_READY and trainer is not None:
    smoke_rows = EVAL_ROWS[:5]
    smoke_outputs = []
    model.eval()
    for row in smoke_rows:
        if row["messages"][-1]["role"] != "assistant":
            continue
        pred = generate_final(model, row["messages"][:-1])
        record = {
            "id": row["id"],
            "category": row.get("category"),
            "prediction_final_channel": pred,
            "prediction_is_byv_fp": pred.startswith(HEADER + "\n"),
        }
        smoke_outputs.append(record)
        print("\n=====", row["id"], row.get("category"), "=====")
        print(pred[:1200])
    (EXPERIMENT_DIR / "eval_smoke_outputs.jsonl").write_text(
        ''.join(json.dumps(x, ensure_ascii=False) + "\n" for x in smoke_outputs), encoding="utf-8"
    )
else:
    print("Smoke test skipped because training did not run.")

## 9. Simpan manifest dan cek artefak

Adapter hasil training adalah LoRA adapter, bukan model merged atau GGUF. Proses merge/quantization/deployment perlu benchmark tersendiri.

In [ ]:
import importlib.metadata as metadata
package_versions = {}
for pkg in ["torch", "transformers", "datasets", "accelerate", "peft", "bitsandbytes", "safetensors"]:
    try:
        package_versions[pkg] = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        package_versions[pkg] = None

manifest = {
    "run_name": RUN_NAME,
    "base_model": MODEL_ID,
    "base_model_revision": BASE_MODEL_COMMIT,
    "dataset_version": DATASET_VERSION,
    "protocol": "BYV-FP/1.0",
    "method": "SFT + QLoRA",
    "quantization": "4-bit NF4 + double quantization",
    "compute_dtype": str(COMPUTE_DTYPE),
    "thinking_mode": "Qwen3-4B-Thinking-2507; final-only labels use </think> to close an empty thinking block; no CoT targets included",
    "train_sha256": EXPECTED_DATA_SHA256["train.jsonl"],
    "eval_sha256": EXPECTED_DATA_SHA256["eval.jsonl"],
    "train_records": len(TRAIN_ROWS),
    "eval_records": len(EVAL_ROWS),
    "train_assistant_turns": len(TRAIN_DATASET),
    "eval_assistant_turns": len(EVAL_DATASET),
    "skipped_long_train_targets": len(TRAIN_SKIPPED),
    "skipped_long_eval_targets": len(EVAL_SKIPPED),
    "test_split_loaded": False,
    "gpu_name": GPU_NAME,
    "gpu_vram_gb": GPU_VRAM_GB,
    "gpu_compute_capability": GPU_COMPUTE_CAPABILITY,
    "max_length": MAX_LENGTH,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "training_completed": bool(TRAINING_READY and trainer is not None and trainer.state.global_step > 0),
    "packages": package_versions,
}
(EXPERIMENT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Manifest:", EXPERIMENT_DIR / "manifest.json")
print("Adapter config exists:", (OUTPUT_DIR / "adapter_config.json").exists())
print("Training test split loaded:", manifest["test_split_loaded"])

## 10. Langkah berikutnya

1. Periksa `train_metrics.json`, `manifest.json`, dan `eval_smoke_outputs.jsonl` di Google Drive.
2. Jalankan notebook **BYV-AI_Test_Evaluation_BYV-FP_v0.2.1_Qwen3-4B-Thinking-2507.ipynb** untuk mengukur held-out test 35 contoh.
3. Periksa isi output secara manual; parser/schema validity bukan semantic accuracy.
4. Jangan merge adapter ke base atau melakukan export GGUF sebelum inference end-to-end diuji.

Dokumentasi model dan template: [Qwen3-4B-Thinking-2507](https://huggingface.co/Qwen/Qwen3-4B-Thinking-2507), [Qwen3 quickstart](https://github.com/QwenLM/Qwen3/blob/main/docs/source/getting_started/quickstart.md). Dataset v0.2.1 tetap synthetic seed; tambah contoh human-authored sebelum klaim kualitas umum.